In [8]:
import os
import glob
import pandas as pd

DATA_ROOT = os.path.join("data", "ami")

AUDIO_DIR = os.path.join(DATA_ROOT, "audio")
SUMMARY_DIR = os.path.join(DATA_ROOT, "summary")
TRANSCRIPT_DIR = os.path.join(DATA_ROOT, "transcript")

In [9]:
def load_transcripts(transcript_dir):

    records = []

    for split in ["train", "valid", "test"]:
        split_dir = os.path.join(transcript_dir, split)

        for filepath in glob.glob(os.path.join(split_dir, "*.txt")):
            meeting_id = os.path.basename(filepath).replace(".txt", "")

            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()

            records.append({
                "meeting_id": meeting_id,
                "split": split,
                "transcript": text
            })

    return pd.DataFrame(records)

In [10]:
def load_summaries(summary_dir):

    summaries = {}

    for filepath in glob.glob(os.path.join(summary_dir, "*.txt")):
        meeting_id = os.path.basename(filepath).replace(".txt", "")

        with open(filepath, "r", encoding="utf-8") as f:
            summaries[meeting_id] = f.read()

    return summaries

In [11]:
def load_audio_paths(audio_dir):

    audio_records = []

    for meeting in os.listdir(audio_dir):

        meeting_audio_dir = os.path.join(audio_dir, meeting, "audio")

        if not os.path.isdir(meeting_audio_dir):
            continue

        headset = None
        lapel = None

        for file in os.listdir(meeting_audio_dir):

            if "Headset" in file:
                headset = os.path.join(meeting_audio_dir, file)

            if "Lapel" in file:
                lapel = os.path.join(meeting_audio_dir, file)

        audio_records.append({
            "meeting_id": meeting,
            "audio_headset": headset,
            "audio_lapel": lapel
        })

    return pd.DataFrame(audio_records)

In [12]:
def build_dataset():
    transcripts_df = load_transcripts(TRANSCRIPT_DIR)

    summaries = load_summaries(SUMMARY_DIR)

    audio_df = load_audio_paths(AUDIO_DIR)

    # attach summaries
    transcripts_df["summary"] = transcripts_df["meeting_id"].map(summaries)

    # merge audio paths
    df = transcripts_df.merge(audio_df, on="meeting_id", how="left")

    return df

In [13]:
df = build_dataset()

print(df.shape)
print(df.head())

(137, 6)
  meeting_id  split                                         transcript  \
0    ES2002a  train  B\t50.42\tOkay .\tintroduction of participants...   
1    ES2002b  train  B\t14.59\tIs that alright now ?\topening\nB\t1...   
2    ES2002c  train  A\t0.02\t'S to do now is to decide how to fulf...   
3    ES2002d  train  B\t2.41\tOkay we all all set ?\topening\nB\t5....   
4    ES2005a  train  C\t0.08\tUh , making a profit of fifty million...   

                                             summary  \
0  The project manager introduced the upcoming pr...   
1  The project manager briefed the team on some n...   
2  The project manager recapped the decisions mad...   
3  The project manager recapped the decisions mad...   
4  The group discussed their initial ideas about ...   

                                       audio_headset  \
0  data\ami\audio\ES2002a\audio\ES2002a.Mix-Heads...   
1  data\ami\audio\ES2002b\audio\ES2002b.Mix-Heads...   
2  data\ami\audio\ES2002c\audio\ES2002c.M